In [1]:
import pybryt

In [2]:
# 範例：比較「有沒有加 User-Agent」的差別（已完成，供參考）
import requests


def show_headers(url, headers=None, label=""):
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        print(f"{label}伺服器收到的 headers：")
        print(resp.json())
    except Exception as e:
        print(f"{label}請求失敗（httpbin.org 可能暫時連不上）：{e}")


show_headers("https://httpbin.org/headers", label="沒加 User-Agent，")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
show_headers("https://httpbin.org/headers", headers=headers, label="加上 User-Agent 後，")

沒加 User-Agent，伺服器收到的 headers：
沒加 User-Agent，請求失敗（httpbin.org 可能暫時連不上）：Expecting value: line 1 column 1 (char 0)
加上 User-Agent 後，伺服器收到的 headers：
加上 User-Agent 後，請求失敗（httpbin.org 可能暫時連不上）：Expecting value: line 1 column 1 (char 0)


In [3]:
def fetch_with_header(url):
    """建立帶有 User-Agent 的 headers，發送請求並回傳 status_code"""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers)
    return response.status_code


# ===== 自我檢查 =====
from unittest.mock import patch, MagicMock


def _fake_get(url, headers=None, **kwargs):
    mock_resp = MagicMock()
    mock_resp.status_code = 200
    _fake_get.captured_headers = headers
    return mock_resp


with patch("requests.get", side_effect=_fake_get):
    check_status = fetch_with_header("https://example.com")

check_headers = getattr(_fake_get, "captured_headers", None) or {}
check_ua = check_headers.get("User-Agent", "")

assert check_status == 200, "函式沒有正確回傳 response 的狀態碼，請確認有 return response.status_code。"
assert "User-Agent" in check_headers, "headers 字典裡沒有找到 User-Agent 這個 key，請檢查拼字（大小寫要對）。"
assert "python-requests" not in check_ua.lower() and any(
    keyword in check_ua.lower() for keyword in ("mozilla", "chrome", "safari", "firefox", "edge")
), "User-Agent 字串不像瀏覽器（或還是預設值），請參考題目提示的格式。"

print("恭喜通過！你已經學會如何用 User-Agent 偽裝成瀏覽器")

恭喜通過！你已經學會如何用 User-Agent 偽裝成瀏覽器


In [1]:
with patch("requests.get", side_effect=_fake_get):
    submit_status = fetch_with_header("https://example.com")

submit_headers = getattr(_fake_get, "captured_headers", None) or {}
submit_ua = submit_headers.get("User-Agent", "")

status_is_valid = (submit_status == 200)
has_user_agent_key = "User-Agent" in submit_headers
user_agent_is_valid = (
    "python-requests" not in submit_ua.lower()
    and any(keyword in submit_ua.lower() for keyword in ("mozilla", "chrome", "safari", "firefox", "edge"))
)

# 用獨一無二的簽章 tuple 包住布林值，避免裸的 True 在執行足跡裡太常見而誤判
status_signature = ("chapter-2-level-3-1:status_is_valid", status_is_valid)
has_ua_signature = ("chapter-2-level-3-1:has_user_agent_key", has_user_agent_key)
ua_format_signature = ("chapter-2-level-3-1:user_agent_is_valid", user_agent_is_valid)

pybryt.Value(status_signature, name="status_is_valid",
    success_message="函式正確回傳了 response 的狀態碼！",
    failure_message="函式沒有正確回傳 response 的狀態碼。")

pybryt.Value(has_ua_signature, name="has_user_agent_key",
    success_message="headers 裡有帶入 User-Agent！",
    failure_message="headers 字典裡沒有找到 User-Agent 這個 key。")

pybryt.Value(ua_format_signature, name="user_agent_is_valid",
    success_message="User-Agent 字串看起來像瀏覽器，成功偽裝！",
    failure_message="User-Agent 字串不像瀏覽器，請參考題目提示的格式。")

NameError: name 'patch' is not defined